# Notebook 6: Metadata Extraction with LangExtract

**Difficulty:** Intermediate | **Estimated Time:** 45 minutes

## Learning Objectives

1. ✅ Explain how LangExtract differs from the LlamaIndex metadata extractors
2. ✅ Define an extraction task with a prompt and one example
3. ✅ Run LangExtract with the Groq LLM
4. ✅ Check that each value links to its place in the source text
5. ✅ Store the values as node metadata and filter retrieval with them

## LangExtract and the LlamaIndex extractors

| Item | `SummaryExtractor`, `TitleExtractor` | LangExtract |
|---|---|---|
| Output | Free text (a summary or a title) | Typed values (classes and attributes) |
| You define | Nothing, or a prompt template | A prompt and examples |
| Source link | None | The exact text and its character position |
| Best for | Context for each chunk | Filters and structured fields |

> **Cost:** this notebook makes about 3 LLM calls. It waits between calls because of the Groq limit of 8000 tokens each minute.

---

## 1. Setup

In [ ]:
import os
import sys
import time
import textwrap
from pathlib import Path

from dotenv import load_dotenv
import langextract as lx

from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.core.schema import TextNode
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator
from llama_index.embeddings.fastembed import FastEmbedEmbedding

# Find the project root and read the model name from config.yaml
project_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "project_config.py").exists())
sys.path.insert(0, str(project_root))
from project_config import llm_model

load_dotenv()
Settings.embed_model = FastEmbedEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.llm = None  # this notebook retrieves only, so it needs no LlamaIndex LLM

print("✅ Setup complete")
print(f"  Model: {llm_model()}")

### Connect LangExtract to Groq

LangExtract has providers for Gemini, OpenAI and Ollama. Groq has an OpenAI-compatible API.
Use the `OpenAILanguageModel` provider with the Groq `base_url`.

In [ ]:
model_config = lx.factory.ModelConfig(
    model_id=llm_model(),
    provider="OpenAILanguageModel",
    provider_kwargs={
        "api_key": os.environ["GROQ_API_KEY"],
        "base_url": "https://api.groq.com/openai/v1",
    },
)
print("✅ LangExtract model configured")

---

## 2. Load the Document

In [ ]:
documents = SimpleDirectoryReader(
    input_files=[str(project_root / "data/research_papers/docling_technical_report_2024.pdf")]
).load_data()   # one Document for each page

print(f"Pages loaded: {len(documents)}")
print(f"Page 3 length: {len(documents[2].text)} characters")

---

## 3. Define the Extraction Task

An extraction task has two parts:

1. **A prompt** that says what to extract.
2. **Examples** that show the output format. LangExtract needs at least one example.

The example below shows three classes: `model`, `dataset` and `software`. The `attributes` field holds extra facts.

In [ ]:
prompt = textwrap.dedent("""\
    Extract software components, AI models, and datasets from the text.
    Use the exact text from the document. Do not paraphrase.""")

examples = [
    lx.data.ExampleData(
        text="The pipeline uses RT-DETR for layout analysis. It was trained on the DocLayNet dataset. The model runs with the onnxruntime library.",
        extractions=[
            lx.data.Extraction(
                extraction_class="model",
                extraction_text="RT-DETR",
                attributes={"task": "layout analysis"},
            ),
            lx.data.Extraction(
                extraction_class="dataset",
                extraction_text="DocLayNet",
            ),
            lx.data.Extraction(
                extraction_class="software",
                extraction_text="onnxruntime",
                attributes={"type": "library"},
            ),
        ],
    )
]
print("✅ Prompt and example defined")

---

## 4. Extract from One Page

Two options are important with Groq:

- `fence_output=True`: the model returns the JSON in a code fence.
- `use_schema_constraints=False`: Groq models do not use the LangExtract schema constraint.

In [ ]:
def extract_page(text: str):
    result = lx.extract(
        text_or_documents=text,
        prompt_description=prompt,
        examples=examples,
        config=model_config,
        fence_output=True,
        use_schema_constraints=False,
        max_char_buffer=4000,   # keep each request small for the Groq token limit
    )
    time.sleep(20)  # stay below the Groq limit of 8000 tokens each minute
    return result

page_text = documents[2].text   # page 3
result = extract_page(page_text)

for item in result.extractions:
    print(f"{item.extraction_class:8} | {item.extraction_text}")
    print(f"           attributes: {item.attributes}")

### Check the Source Link

Each extraction has a `char_interval`. It gives the start and end position in the page text.
Use it to check that the value comes from the document and not from the model memory.

In [ ]:
for item in result.extractions:
    interval = item.char_interval
    if interval is None:
        print(f"⚠️  No source link: {item.extraction_text}")
        continue
    quoted = page_text[interval.start_pos:interval.end_pos]
    status = "✅" if quoted == item.extraction_text else "⚠️ "
    print(f"{status} [{interval.start_pos}:{interval.end_pos}] {quoted!r}")

---

## 5. Extract from Several Pages and Store the Values as Metadata

LlamaIndex metadata values must be simple types. Join the values of each class into one comma-separated string.

In [ ]:
pages_to_process = [2, 3, 4]   # pages 3, 4 and 5

for index in pages_to_process:
    document = documents[index]
    page_result = result if index == 2 else extract_page(document.text)

    for cls, key in (("model", "models"), ("dataset", "datasets"), ("software", "software")):
        values = sorted({e.extraction_text for e in page_result.extractions if e.extraction_class == cls})
        document.metadata[key] = ", ".join(values)

    print(f"Page {document.metadata['page_label']}:")
    print(f"  models:   {document.metadata['models']}")
    print(f"  datasets: {document.metadata['datasets']}")
    print(f"  software: {document.metadata['software']}")

---

## 6. Filter Retrieval with the Extracted Metadata

1. Make one node for each processed page. Each node keeps its metadata.
2. Index the nodes.
3. Retrieve with and without a metadata filter.

`FilterOperator.TEXT_MATCH` finds a word inside the metadata string. This step makes no LLM call.

In [ ]:
nodes = [
    TextNode(text=documents[i].text, metadata=documents[i].metadata)
    for i in pages_to_process
]
index = VectorStoreIndex(nodes=nodes)

question = "How does the system recognize table structure?"

print("Without a filter:")
for hit in index.as_retriever(similarity_top_k=3).retrieve(question):
    print(f"  page {hit.node.metadata['page_label']} | score {hit.score:.3f} | models: {hit.node.metadata['models']}")

table_filter = MetadataFilters(filters=[
    MetadataFilter(key="models", value="TableFormer", operator=FilterOperator.TEXT_MATCH)
])
print("\nWith the filter models contains 'TableFormer':")
for hit in index.as_retriever(similarity_top_k=3, filters=table_filter).retrieve(question):
    print(f"  page {hit.node.metadata['page_label']} | score {hit.score:.3f} | models: {hit.node.metadata['models']}")

---

## 7. Key Takeaways

1. **LangExtract gives typed values.** The LlamaIndex extractors give free text.
2. **Examples control the output.** Add more examples and more classes to extract other fields.
3. **Each value has a source link.** Check `char_interval` to find values that the model invented.
4. **Join lists into strings** before you store them as LlamaIndex metadata.
5. **Use a filter** (`TEXT_MATCH`) to limit retrieval to nodes with a given value.
6. **Mind the Groq limits.** Use a small `max_char_buffer` and wait between calls.

### Next Steps

- Add a class such as `person` or `organization` to the prompt and the example.
- Use `extraction_passes=2` for better recall. This uses more tokens.
- Compare the result with `TitleExtractor` and `SummaryExtractor` in Notebook 2.